# 4. Endpoints (Models)

Welcome to the Endpoints notebook! 👋

In this guide, we'll walk you through managing **endpoints** in Enkrypt AI. Think of endpoints as the bridge between Enkrypt AI and your AI applications – they're the URLs where your AI models live and serve requests.

Whether you're protecting an OpenAI endpoint, an internal LLM deployment, or a custom AI application, Enkrypt AI makes it easy to add guardrails and policies to any model endpoint.

## What You'll Learn

By the end of this notebook, you'll know how to:
- ✅ Add a new model endpoint to Enkrypt AI
- ✅ Check the health of your endpoints
- ✅ Retrieve and manage endpoint details
- ✅ List all your configured endpoints
- ✅ Modify existing endpoints
- ✅ Clean up by deleting endpoints

Let's get started! 🚀


## Setup

First, let's import the necessary libraries and initialize our Enkrypt AI clients. Make sure you have your `ENKRYPTAI_API_KEY` and `OPENAI_API_KEY` set in your `.env` file.


In [ ]:
import os
import copy
from enkryptai_sdk import *
from dotenv import load_dotenv

load_dotenv()

# Environment Variables
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ENKRYPT_API_KEY = os.getenv("ENKRYPTAI_API_KEY")
ENKRYPT_BASE_URL = os.getenv("ENKRYPTAI_BASE_URL") or "https://api.enkryptai.com"

# Initialize Clients
model_client = ModelClient(api_key=ENKRYPT_API_KEY, base_url=ENKRYPT_BASE_URL)
redteam_client = RedTeamClient(api_key=ENKRYPT_API_KEY, base_url=ENKRYPT_BASE_URL)

print("✅ Enkrypt AI clients initialized successfully!")


## Configuration Variables

Let's set up some variables for our model endpoint. We'll use OpenAI's GPT-4 as an example, but you can adapt this for any AI model or application endpoint.


In [ ]:
# Model identification
test_model_saved_name = "my-gpt4-endpoint"  # A friendly name for your endpoint
test_model_version = "v1"  # Version control for your endpoints
model_name = "gpt-4"  # The actual model name
model_provider = "openai"  # The provider (openai, anthropic, etc.)

# Endpoint configuration
model_endpoint_url = "https://api.openai.com/v1/chat/completions"  # Your model's endpoint URL

# Verify API key is loaded
if not OPENAI_API_KEY:
    print("⚠️  Warning: OPENAI_API_KEY not found in environment variables")
else:
    print("✅ API key loaded successfully")


## Sample Model Configuration

Here's how we structure a model configuration. This tells Enkrypt AI everything it needs to know about your endpoint:

- **model_saved_name**: Your custom name for this endpoint
- **model_version**: Version control (helpful for A/B testing or gradual rollouts)
- **testing_for**: Type of model (typically "foundationModels")
- **model_name**: The actual model identifier
- **model_config**: Detailed configuration including provider, URL, API key, and modalities


In [ ]:
sample_model_config = {
    "model_saved_name": test_model_saved_name,
    "model_version": test_model_version,
    "testing_for": "foundationModels",
    "model_name": model_name,
    "model_config": {
        "model_provider": model_provider,
        "endpoint_url": model_endpoint_url,
        "apikey": OPENAI_API_KEY,
        "input_modalities": ["text"],
        "output_modalities": ["text"],
    },
}

print("✅ Model configuration created!")
print(f"\nEndpoint name: {sample_model_config['model_saved_name']}")
print(f"Version: {sample_model_config['model_version']}")
print(f"Model: {sample_model_config['model_name']}")


## 1. Adding a Model Endpoint

Now comes the exciting part – let's add your first model endpoint to Enkrypt AI! This registers your AI application with our platform so you can start applying guardrails and policies.


In [ ]:
# Add the model endpoint
add_model_response = model_client.add_model(config=copy.deepcopy(sample_model_config))

print(add_model_response)
print(f"\n✅ Status: {add_model_response.message}")

# You can also view the response as a dictionary
print("\nFull response:")
print(add_model_response.to_dict())


## 2. Check Model Health

After adding an endpoint, it's always a good idea to check its health. This ensures Enkrypt AI can successfully communicate with your AI model.


In [ ]:
# Check if the endpoint is healthy
check_saved_model_health = redteam_client.check_saved_model_health(
    model_saved_name=test_model_saved_name, 
    model_version=test_model_version
)

print(check_saved_model_health)
print(f"\n✅ Health Status: {check_saved_model_health.status}")


## 3. Retrieve Model Details

Need to see the configuration of a specific endpoint? You can easily retrieve all the details you need.


In [ ]:
# Get detailed information about your endpoint
model_details = model_client.get_model(
    model_saved_name=test_model_saved_name, 
    model_version=test_model_version
)

print(model_details)

# Access specific fields
print("\n📋 Endpoint Details:")
print(f"  Saved Name: {model_details.model_saved_name}")
print(f"  Version: {model_details.model_version}")
print(f"  Model Name: {model_details.model_name}")
print(f"  Provider: {model_details.model_config.model_provider}")

# View complete configuration as dictionary
print("\nFull configuration:")
print(model_details.to_dict())


## 4. List All Model Endpoints

As your AI infrastructure grows, you might have multiple endpoints. Let's see how to list them all at once.


In [ ]:
# Get a list of all your configured endpoints
models = model_client.get_model_list()

print(models)

# Get the first model
print("\nFirst model:")
print(models.models[0])
print(f"Model name: {models.models[0].model_name}")

# Get the last model
print("\nLast model:")
print(models.models[-1])
print(f"Model name: {models.models[-1].model_name}")

# Print as a dictionary
print("\nAll models as dictionary:")
print(models.to_dict())


## 5. Modify a Model Endpoint

Need to update your endpoint configuration? Maybe you want to switch to a different model version or update API credentials. Here's how!

**💡 Pro Tip:** Instead of modifying existing endpoints, consider creating a new version. This way, you can test changes without breaking existing integrations. You can always delete the old version later.


In [ ]:
# Create a modified configuration
new_model_config = copy.deepcopy(sample_model_config)

# Example: Switch to a different model
new_model_config["model_name"] = "gpt-4o-mini-test"

# If you want to change the saved name or version (creates a new endpoint):
# new_model_config["model_saved_name"] = "my-gpt4-endpoint-v2"
# new_model_config["model_version"] = "v2"

# Determine if we're changing the identifier
old_model_saved_name = None
if new_model_config["model_saved_name"] != test_model_saved_name:
    old_model_saved_name = test_model_saved_name

old_model_version = None
if new_model_config["model_version"] != test_model_version:
    old_model_version = test_model_version

# Apply the modification
modify_response = model_client.modify_model(
    old_model_saved_name=old_model_saved_name,
    old_model_version=old_model_version,
    config=new_model_config
)

print(modify_response)
print(f"\n✅ Status: {modify_response.message}")
print("\nFull response:")
print(modify_response.to_dict())


## 6. Delete a Model Endpoint

When you're done with an endpoint or want to clean up your configuration, you can easily remove it.

**⚠️  Important:** Make sure no active policies or deployments are using this endpoint before deleting it!


In [ ]:
# Delete the endpoint
delete_response = model_client.delete_model(
    model_saved_name=test_model_saved_name,
    model_version=test_model_version
)

print(delete_response)
print(f"\n✅ Status: {delete_response.message}")
print("\nFull response:")
print(delete_response.to_dict())


## Summary

Congratulations! 🎉 You've learned how to manage model endpoints in Enkrypt AI.

### What You've Accomplished:

- ✅ Added a new AI model endpoint
- ✅ Verified endpoint health
- ✅ Retrieved and inspected endpoint details
- ✅ Listed all configured endpoints
- ✅ Modified endpoint configurations
- ✅ Cleaned up by deleting endpoints

### Next Steps:

Now that you know how to manage endpoints, you can:
1. **Apply Policies:** Use the policies from notebook 2 to protect your endpoints
2. **Add Guardrails:** Implement real-time safety checks (notebook 3)
3. **Red Team:** Test your endpoints for vulnerabilities (notebook 5)
4. **Deploy:** Push your protected endpoints to production (notebook 6)

### Quick Tips:

- 💡 Use version control for your endpoints to enable safe A/B testing
- 💡 Always check endpoint health after configuration changes
- 💡 Keep your API keys secure and rotate them regularly
- 💡 Document your endpoint configurations for your team

Ready to move forward? Head to **notebook 5** to learn about red teaming your AI applications! 🚀


---

### Need Help?

- 📖 [Documentation](https://docs.enkryptai.com)
- 💬 [Contact Support](https://enkryptai.com/request-a-demo)
- 🌐 [Website](https://enkryptai.com)

**Ship Fast. Ship Safe. Stay Ahead.**
